In [290]:
import numpy as np 
import pandas as pd

import plotly.express as px

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
pd.set_option('display.max_columns', None)

In [291]:
#lese inn data fra demographic
df_demographic = pd.read_csv("raw_data/demographic.csv")
df_demographic.head()


,pasient_id,alder,kjønn,utdanning,inntekt,etnisitet
0,2,60.33899,female,12.0,$11-$25k,white
1,3,52.74698,female,12.0,under $11k,white
2,4,42.38498,female,11.0,under $11k,white
3,5,79.88495,female,NaN,NaN,white
4,6,93.01599,male,14.0,NaN,white


In [292]:
#leser inn data fra hospital 

df_hospital = pd.read_csv("raw_data/hospital.csv")
df_hospital.head()


,pasient_id,sykehusdød,oppholdslengde
0,2,1,4
1,3,0,17
2,4,0,3
3,5,0,-99
4,6,1,4


In [293]:
#leser inn data fra physiological
df_physiological = pd.read_csv("raw_data/physiological.txt", delimiter='\t')
df_physiological.head()

,pasient_id,blodtrykk,hvite_blodlegemer,hjertefrekvens,respirasjonsfrekvens,kroppstemperatur,lungefunksjon,serumalbumin,bilirubin,kreatinin,natrium,blod_ph,glukose,blodurea_nitrogen,urinmengde
0,2,43.0,17.097656,112.0,34.0,34.59375,98.00000,NaN,NaN,5.500000,132.0,7.250000,NaN,NaN,NaN
1,3,70.0,8.500000,88.0,28.0,37.39844,231.65625,NaN,2.199707,2.000000,134.0,7.459961,NaN,NaN,NaN
2,4,75.0,9.099609,88.0,32.0,35.00000,NaN,NaN,NaN,0.799927,139.0,NaN,NaN,NaN,NaN
3,5,59.0,13.500000,112.0,20.0,37.89844,173.31250,NaN,NaN,0.799927,143.0,7.509766,NaN,NaN,NaN
4,6,110.0,10.398438,101.0,44.0,38.39844,266.62500,NaN,NaN,0.699951,140.0,7.659180,NaN,NaN,NaN


In [294]:
#leser inn data fra severity
df_severity = pd.read_json("raw_data/severity.json")
df_severity.head()

,sykdomskategori_id,sykdomskategori,pasient_id,dødsfall,sykdom_underkategori,antall_komorbiditeter,koma_score,adl_pasient,adl_stedfortreder,fysiologisk_score,apache_fysiologisk_score,overlevelsesestimat_2mnd,overlevelsesestimat_6mnd,diabetes,demens,kreft,lege_overlevelsesestimat_2mnd,lege_overlevelsesestimat_6mnd,dnr_status,dnr_dag
0,A1s,ARF/MOSF,"[5, 15, 18, 23, 28, 34, 39, 43, 46, 47, 48, 58...","[0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, ...","[ARF/MOSF w/Sepsis, ARF/MOSF w/Sepsis, ARF/MOS...","[1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 2, 0, 3, 1, 3, ...","[26.0, 26.0, 26.0, 0.0, 26.0, 37.0, 0.0, 0.0, ...","[None, None, None, None, None, None, None, Non...","[2.0, None, 0.0, 5.0, 2.0, None, 0.0, None, No...","[23.5, 30.5, 40.296875, 31.6992188, 46.796875,...","[30.0, 39.0, 58.0, 42.0, 85.0, 49.0, 5.0, 76.0...","[0.6348876950000001, 0.590942383, 0.2129821780...","[0.5329589840000001, 0.481994629, 0.1169891360...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[no, no, no, no, yes, no, no, no, no, no, yes,...","[0.899999619, 0.899999619, 0.09999996400000001...","[0.9, 0.9, 0.001, 0.5, 0.000125, 0.60000000000...","[None, None, None, None, None, None, None, Non...","[None, None, None, None, None, None, None, Non..."
1,BrY,COPD/CHF/Cirrhosis,"[2, 3, 7, 8, 11, 13, 14, 19, 20, 30, 31, 32, 3...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, ...","[Cirrhosis, Cirrhosis, CHF, CHF, CHF, Cirrhosi...","[2, 2, 1, 3, 1, 1, 0, 2, 1, 2, 2, 2, 2, 1, 1, ...","[44.0, 0.0, 0.0, 26.0, 0.0, 0.0, 0.0, 0.0, 0.0...","[None, 1.0, 0.0, None, 2.0, 0.0, 0.0, 7.0, 3.0...","[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 7.0, None,...","[52.6953125, 20.5, 17.296875, 21.5976562, 14.5...","[74.0, 45.0, 46.0, 53.0, 14.0, 30.0, 34.0, 42....","[0.000999928, 0.790893555, 0.892944336, 0.6708...","[0.0, 0.6649169920000001, 0.820922852, 0.49896...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[no, no, no, no, no, no, no, no, no, no, no, n...","[0.0, 0.75, None, 0.799999714, 0.699999809, No...","[0.0, 0.5, 0.7000000000000001, 0.4, 0.5, None,...","[None, None, None, None, None, None, None, Non...","[None, None, None, None, None, None, None, Non..."
2,ChE,Cancer,"[4, 9, 10, 12, 16, 17, 21, 24, 27, 41, 42, 54,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ...","[Lung Cancer, Lung Cancer, Colon Cancer, Lung ...","[2, 2, 0, 0, 1, 2, 0, 0, 0, 1, 1, 1, 2, 0, 1, ...","[0.0, 26.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 44.0...","[0.0, None, 0.0, 1.0, 2.0, None, 0.0, None, No...","[0.0, 7.0, None, 1.0, 0.0, None, 0.0, None, No...","[20.0976562, 15.8984375, 2.2998047, 16.3984375...","[19.0, 17.0, 9.0, 17.0, 11.0, 4.0, 16.0, 11.0,...","[0.6989746090000001, 0.570922852, 0.9528808590...","[0.411987305, 0.24899292, 0.8879394530000001, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[metastatic, metastatic, metastatic, metastati...","[0.899999619, 0.049999982000000005, None, None...","[0.5, 0.000125, None, None, 0.7000000000000001...","[None, dnr ved innleggelse, None, None, None, ...","[None, 0.0, None, None, None, None, None, None..."
3,DWw,Coma,"[6, 162, 188, 250, 252, 262, 275, 309, 323, 35...","[1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, ...","[Coma, Coma, Coma, Coma, Coma, Coma, Coma, Com...","[1, 0, 0, 2, 1, 2, 1, 0, 0, 1, 1, 0, 2, 2, 0, ...","[55.0, 61.0, 94.0, 55.0, 94.0, 100.0, 100.0, 5...","[None, None, None, None, None, None, None, Non...","[1.0, None, None, 1.0, None, None, None, None,...","[19.3984375, 30.3984375, 20.296875, 30.8984375...","[27.0, 36.0, 22.0, 53.0, 40.0, 25.0, 58.0, 16....","[0.28497314500000004, 0.438964844, 0.280944824...","[0.214996338, 0.365966797, 0.211975098, 0.2729...","[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...","[no, no, no, no, yes, no, no, no, no, no, no, ...","[0.0, None, None, None, None, 0.09999996400000...","[0.0, None, N

Konverter alle om til float i demographic

In [295]:


df_demographic['kjønn'] = pd.get_dummies(df_demographic['kjønn'], drop_first=True)

df_demographic = pd.get_dummies(df_demographic, columns=['etnisitet'], drop_first=True)

income_mapping = {
    '$11-$25k': 1,
    'under $11k': 0,
    '$25k-$50k': 2,
    '$50k-$75k': 3,
    'above $75k': 4
}
df_demographic['inntekt'] = df_demographic['inntekt'].map(income_mapping)

#Gjør alle om til float
df_demographic = df_demographic.astype(float)

df_demographic.head()

,pasient_id,alder,kjønn,utdanning,inntekt,etnisitet_black,etnisitet_hispanic,etnisitet_other,etnisitet_white
0,2.0,60.33899,0.0,12.0,1.0,0.0,0.0,0.0,1.0
1,3.0,52.74698,0.0,12.0,0.0,0.0,0.0,0.0,1.0
2,4.0,42.38498,0.0,11.0,0.0,0.0,0.0,0.0,1.0
3,5.0,79.88495,0.0,NaN,NaN,0.0,0.0,0.0,1.0
4,6.0,93.01599,1.0,14.0,NaN,0.0,0.0,0.0,1.0


datavasking av hospital 

In [296]:

#fjerner alle med nagativ oppholdslengde
df_hospital = df_hospital[df_hospital['oppholdslengde'] >= 0]



Datavasking av severity 

In [297]:
#legge det opp slik at hver pasient for riktig 
df_severity = df_severity.iloc[:, 0:-1].explode(list(df_severity.columns[2:-1]))
df_severity.reset_index(drop=True, inplace=True)
df_severity=df_severity.sort_values(by=['pasient_id'], ignore_index=True)

df_severity = pd.get_dummies(df_severity, columns=['sykdomskategori_id'])
df_severity = pd.get_dummies(df_severity, columns=['sykdomskategori'])
df_severity = pd.get_dummies(df_severity, columns=['sykdom_underkategori'])
df_severity = pd.get_dummies(df_severity, columns=['kreft'])
df_severity = pd.get_dummies(df_severity, columns=['dnr_status'])





df_severity = df_severity.astype(float)
df_severity.head()

,pasient_id,dødsfall,antall_komorbiditeter,koma_score,adl_pasient,adl_stedfortreder,fysiologisk_score,apache_fysiologisk_score,overlevelsesestimat_2mnd,overlevelsesestimat_6mnd,diabetes,demens,lege_overlevelsesestimat_2mnd,lege_overlevelsesestimat_6mnd,sykdomskategori_id_A1s,sykdomskategori_id_BrY,sykdomskategori_id_ChE,sykdomskategori_id_DWw,sykdomskategori_ARF/MOSF,sykdomskategori_COPD/CHF/Cirrhosis,sykdomskategori_Cancer,sykdomskategori_Coma,sykdom_underkategori_ARF/MOSF w/Sepsis,sykdom_underkategori_CHF,sykdom_underkategori_COPD,sykdom_underkategori_Cirrhosis,sykdom_underkategori_Colon Cancer,sykdom_underkategori_Coma,sykdom_underkategori_Lung Cancer,sykdom_underkategori_MOSF w/Malig,kreft_metastatic,kreft_no,kreft_yes,dnr_status_dnr før innleggelse,dnr_status_dnr ved innleggelse
0,2.0,1.0,2.0,44.0,NaN,1.0,52.695312,74.0,0.001000,0.000000,0.0,0.0,0.00,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,3.0,1.0,2.0,0.0,1.0,0.0,20.500000,45.0,0.790894,0.664917,0.0,0.0,0.75,0.5,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,4.0,1.0,2.0,0.0,0.0,0.0,20.097656,19.0,0.698975,0.411987,0.0,0.0,0.90,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,5.0,0.0,1.0,26.0,NaN,2.0,23.500000,30.0,0.634888,0.532959,0.0,0.0,0.90,0.9,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,6.0,1.0,1.0,55.0,NaN,1.0,19.398438,27.0,0.284973,0.214996,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [298]:
df_new = pd.DataFrame()

for row in df_severity:
    for column in df_severity.columns:
        pass
df_severity.head(10)

,pasient_id,dødsfall,antall_komorbiditeter,koma_score,adl_pasient,adl_stedfortreder,fysiologisk_score,apache_fysiologisk_score,overlevelsesestimat_2mnd,overlevelsesestimat_6mnd,diabetes,demens,lege_overlevelsesestimat_2mnd,lege_overlevelsesestimat_6mnd,sykdomskategori_id_A1s,sykdomskategori_id_BrY,sykdomskategori_id_ChE,sykdomskategori_id_DWw,sykdomskategori_ARF/MOSF,sykdomskategori_COPD/CHF/Cirrhosis,sykdomskategori_Cancer,sykdomskategori_Coma,sykdom_underkategori_ARF/MOSF w/Sepsis,sykdom_underkategori_CHF,sykdom_underkategori_COPD,sykdom_underkategori_Cirrhosis,sykdom_underkategori_Colon Cancer,sykdom_underkategori_Coma,sykdom_underkategori_Lung Cancer,sykdom_underkategori_MOSF w/Malig,kreft_metastatic,kreft_no,kreft_yes,dnr_status_dnr før innleggelse,dnr_status_dnr ved innleggelse
0,2.0,1.0,2.0,44.0,NaN,1.0,52.695312,74.0,0.001000,0.000000,0.0,0.0,0.00,0.000000,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,3.0,1.0,2.0,0.0,1.0,0.0,20.500000,45.0,0.790894,0.664917,0.0,0.0,0.75,0.500000,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,4.0,1.0,2.0,0.0,0.0,0.0,20.097656,19.0,0.698975,0.411987,0.0,0.0,0.90,0.500000,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,5.0,0.0,1.0,26.0,NaN,2.0,23.500000,30.0,0.634888,0.532959,0.0,0.0,0.90,0.900000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,6.0,1.0,1.0,55.0,NaN,1.0,19.398438,27.0,0.284973,0.214996,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
5,7.0,1.0,1.0,0.0,0.0,1.0,17.296875,46.0,0.892944,0.820923,0.0,0.0,NaN,0.700000,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
6,8.0,1.0,3.0,26.0,NaN,0.0,21.597656,53.0,0.670898,0.498962,1.0,0.0,0.80,0.400000,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,9.0,1.0,2.0,26.0,NaN,7.0,15.898438,17.0,0.570923,0.248993,0.0,1.0,0.05,0.000125,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
8,10.0,1.0,0.0,0.0,0.0,NaN,2.299805,9.0,0.952881,0.887939,0.0,0.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9,11.0,1.0,1.0,0.0,2.0,1.0,14.599609,14.0,0.935913,0.890991,0.0,0.0,0.70,0.500000,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


Finding missing values: 

In [299]:
print("Missing values in df_demographic is:\n", df_demographic.isnull().sum())

Missing values in df_demographic is:
 pasient_id               0
alder                    0
kjønn                    0
utdanning             1382
inntekt               4003
etnisitet_black          0
etnisitet_hispanic       0
etnisitet_other          0
etnisitet_white          0
dtype: int64


In [300]:
print("Missing values in df_hospital is:\n", df_hospital.isnull().sum())


Missing values in df_hospital is:
 pasient_id        0
sykehusdød        0
oppholdslengde    0
dtype: int64


In [301]:
print("Missing values in df_physiological is:\n", df_physiological.isnull().sum())

Missing values in df_physiological is:
 pasient_id                 0
blodtrykk                  0
hvite_blodlegemer        175
hjertefrekvens             0
respirasjonsfrekvens       0
kroppstemperatur           0
lungefunksjon           1944
serumalbumin            2849
bilirubin               2196
kreatinin                 57
natrium                    0
blod_ph                 1912
glukose                 3823
blodurea_nitrogen       3692
urinmengde              4113
dtype: int64


In [302]:
print("Missing values in df_severity is:\n",df_severity.isnull().sum() )


Missing values in df_severity is:
 pasient_id                                   0
dødsfall                                     0
antall_komorbiditeter                        0
koma_score                                   0
adl_pasient                               4798
adl_stedfortreder                         2441
fysiologisk_score                            0
apache_fysiologisk_score                     0
overlevelsesestimat_2mnd                     0
overlevelsesestimat_6mnd                     0
diabetes                                     0
demens                                       0
lege_overlevelsesestimat_2mnd             1421
lege_overlevelsesestimat_6mnd             1407
sykdomskategori_id_A1s                       0
sykdomskategori_id_BrY                       0
sykdomskategori_id_ChE                       0
sykdomskategori_id_DWw                       0
sykdomskategori_ARF/MOSF                     0
sykdomskategori_COPD/CHF/Cirrhosis           0
sykdomskategori_Cancer   

Merger alle tabellene sammen på pasient id

In [303]:
# Slå sammen df_severity og df_demographic
df_merged = pd.merge(df_severity, df_demographic, on='pasient_id', how='inner')

# Slå sammen det midlertidige resultatet df_merged med df_hospital
df_merged = pd.merge(df_merged, df_hospital, on='pasient_id', how='inner')

# Slå sammen det midlertidige resultatet med df_physiological
df_merged = pd.merge(df_merged, df_physiological, on='pasient_id', how='inner')

df_merged = df_merged.astype(float)

df_merged.head()




,pasient_id,dødsfall,antall_komorbiditeter,koma_score,adl_pasient,adl_stedfortreder,fysiologisk_score,apache_fysiologisk_score,overlevelsesestimat_2mnd,overlevelsesestimat_6mnd,diabetes,demens,lege_overlevelsesestimat_2mnd,lege_overlevelsesestimat_6mnd,sykdomskategori_id_A1s,sykdomskategori_id_BrY,sykdomskategori_id_ChE,sykdomskategori_id_DWw,sykdomskategori_ARF/MOSF,sykdomskategori_COPD/CHF/Cirrhosis,sykdomskategori_Cancer,sykdomskategori_Coma,sykdom_underkategori_ARF/MOSF w/Sepsis,sykdom_underkategori_CHF,sykdom_underkategori_COPD,sykdom_underkategori_Cirrhosis,sykdom_underkategori_Colon Cancer,sykdom_underkategori_Coma,sykdom_underkategori_Lung Cancer,sykdom_underkategori_MOSF w/Malig,kreft_metastatic,kreft_no,kreft_yes,dnr_status_dnr før innleggelse,dnr_status_dnr ved innleggelse,alder,kjønn,utdanning,inntekt,etnisitet_black,etnisitet_hispanic,etnisitet_other,etnisitet_white,sykehusdød,oppholdslengde,blodtrykk,hvite_blodlegemer,hjertefrekvens,respirasjonsfrekvens,kroppstemperatur,lungefunksjon,serumalbumin,bilirubin,kreatinin,natrium,blod_ph,glukose,blodurea_nitrogen,urinmengde
0,2.0,1.0,2.0,44.0,NaN,1.0,52.695312,74.0,0.001000,0.000000,0.0,0.0,0.00,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,60.33899,0.0,12.0,1.0,0.0,0.0,0.0,1.0,1.0,4.0,43.0,17.097656,112.0,34.0,34.59375,98.00000,NaN,NaN,5.500000,132.0,7.250000,NaN,NaN,NaN
1,3.0,1.0,2.0,0.0,1.0,0.0,20.500000,45.0,0.790894,0.664917,0.0,0.0,0.75,0.5,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,52.74698,0.0,12.0,0.0,0.0,0.0,0.0,1.0,0.0,17.0,70.0,8.500000,88.0,28.0,37.39844,231.65625,NaN,2.199707,2.000000,134.0,7.459961,NaN,NaN,NaN
2,4.0,1.0,2.0,0.0,0.0,0.0,20.097656,19.0,0.698975,0.411987,0.0,0.0,0.90,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,42.38498,0.0,11.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,75.0,9.099609,88.0,32.0,35.00000,NaN,NaN,NaN,0.799927,139.0,NaN,NaN,NaN,NaN
3,4.0,1.0,2.0,0.0,0.0,0.0,20.097656,19.0,0.698975,0.411987,0.0,0.0,0.90,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,42.38498,0.0,11.0,0.0,0.0,0.0,0.0,1.0,0.0,3.0,75.0,9.099609,88.0,32.0,35.00000,NaN,NaN,NaN,0.799927,139.0,NaN,NaN,NaN,NaN
4,6.0,1.0,1.0,55.0,NaN,1.0,19.398438,27.0,0.284973,0.214996,0.0,0.0,0.00,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,93.01599,1.0,14.0,NaN,0.0,0.0,0.0,1.0,1.0,4.0,110.0,10.398438,101.0,44.0,38.39844,266.62500,NaN,NaN,0.699951,140.0,7.659180,NaN,NaN,NaN


Del inn;
Treningsdata (80 %): Dette er dataene modellen lærer fra. Den justerer sine parametere basert på disse dataene for å finne mønstre og bygge en prediksjonsmodell.
Valideringsdata (10 %): Disse dataene brukes under treningen for å evaluere modellen og justere hyperparametere (for eksempel antall lag i et nevralt nettverk eller dybden på et beslutningstre). Det er viktig at modellen aldri ser disse dataene under selve treningsprosessen for å forhindre overtilpasning.
Testdata (10 %): Dette settet brukes etter at modellen er ferdig trent og validert for å evaluere hvordan modellen presterer på ukjent data. Dette gir en endelig indikator på hvor godt modellen generaliserer til data den aldri har sett før.

In [304]:

X = df_merged.drop(columns=['oppholdslengde']) #feature
y = df_merged['oppholdslengde'] #target-sett

# Anta X og y er dine feature- og target-sett
# Først, trenings- og midlertidig sett
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)

# Deretter, del det midlertidige settet i validering og test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

#1. **Treningsdata**:
   #- **X_train**: Funksjonene (features) for treningsdataene (80 % av de originale dataene).
   #- **y_train**: Målvariabelen (target/labels) for treningsdataene (80 % av de originale dataene).

#2. **Valideringsdata**:
 #  - **X_val**: Funksjonene for valideringsdataene (10 % av de originale dataene).
  # - **y_val**: Målvariabelen for valideringsdataene (10 % av de originale dataene).

#**Testdata**:
# **X_test**: Funksjonene for testdataene (10 % av de originale dataene).
#**y_test**: Målvariabelen for testdataene (10 % av de originale dataene).

### Oppsummering:
#**X_train** og **y_train** inneholder treningsdataene.
#**X_val** og **y_val** inneholder valideringsdataene.
#**X_test** og **y_test** inneholder testdataene.

